## Import

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt


device = 'cuda:2'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data generation

In [6]:
from datasets.Uniform import Uniform

n, d = 10000, 64                                 # < higher d, higher MI
eps = 0.225                                      # < lower eps, higher MI


dataset = Uniform(n_samples=n, n_dims=d//2, eps=eps)
X, Y = dataset.sample_data(n)
X, Y = X.to(device), Y.to(device)
MI = dataset.true_mutual_info()


print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 32.75224627896869


## MI estimation

In [7]:
class Hyperparams(object):
    def __init__(self): 
        self.lr = 5e-4
        self.bs = 500
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [4]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500


finished: t= 0 loss= 1.5187807083129883 loss val= 1.4951014518737793 best val loss= 1.4951014518737793 best t= 0


finished: t= 63 loss= 0.5887722373008728 loss val= 0.5891780853271484 best val loss= 0.4723488986492157 best t= 62


finished: t= 126 loss= 0.4714656174182892 loss val= 0.5238200426101685 best val loss= 0.4581128656864166 best t= 76


finished: t= 189 loss= 0.41371962428092957 loss val= 0.47999176383018494 best val loss= 0.43753159046173096 best t= 128


...

finished: t= 1071 loss= 0.47723910212516785 loss val= 0.5824271440505981 best val loss= 0.4131048917770386 best t= 782


finished: t= 1134 loss= 0.5367100834846497 loss val= 0.5122063159942627 best val loss= 0.4131048917770386 best t= 782


finished: t= 1197 loss= 0.5368194580078125 loss val= 0.4568629860877991 best val loss= 0.4131048917770386 best t= 782




true MI: 32.75224627896869


est MI: 28.421364459991455


In [5]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.0010636746883392334 loss val= 0.0011418312788009644 best val loss= 0.0011418312788009644 best t= 0


finished: t= 63 loss= -6.480559825897217 loss val= -6.60059928894043 best val loss= -6.941038131713867 best t= 45


finished: t= 126 loss= -5.674212455749512 loss val= -6.7622575759887695 best val loss= -7.049090385437012 best t= 121


finished: t= 189 loss= -5.790184020996094 loss val= -6.765798568725586 best val loss= -7.09086799621582 best t= 188


finished: t= 252 loss= -6.4144287109375 loss val= -6.934382438659668 best val loss= -7.09086799621582 best t= 188


finished: t= 315 loss= -6.090226650238037 loss val= -6.617786407470703 best val loss= -7.157783508300781 best t= 290


finished: t= 378 loss= -6.689632892608643 loss val= -6.9056925773620605 best val loss= -7.157783508300781 best t= 290


finished: t= 441 loss= -6.804396152496338 loss val= -6.773477077484131 best val loss= -7.157783508300781 best t= 290




true MI: 32.75224627896869


est MI: 8.942333221435547


In [8]:
## Neural adaptive MI estimate
from estimators.VCE import VCE

estimator = VCE(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.333390235900879 loss val= 1.327141284942627 best val loss= 1.327141284942627 best t= 0
finished: t= 126 loss= 0.4529153108596802 loss val= 0.4772561192512512 best val loss= 0.4533614218235016 best t= 49


finished: t= 0 loss= 1.3500691652297974 loss val= 1.3414710760116577 best val loss= 1.3414710760116577 best t= 0
finished: t= 126 loss= 0.5272184610366821 loss val= 0.5309736728668213 best val loss= 0.4974493980407715 best t= 100
finished: t= 252 loss= 0.5523486733436584 loss val= 0.5210036039352417 best val loss= 0.4910840094089508 best t= 180
finished: t= 378 loss= 0.4919700622558594 loss val= 0.5290707349777222 best val loss= 0.4910840094089508 best t= 180


finished: t= 0 loss= 586.5767211914062 loss val= 591.6901245117188 best val loss= 591.6901245117188 best t= 0
finished: t= 251 loss= 66.4493179321289 loss val= 68.32301330566406 best val loss= 68.32301330566406 best t= 251
finished: t= 502 loss= 65.906982421875 loss val= 68.28483581542969 best val loss= 6

In [6]:
## MI estimate via flows (MIENF)
from estimators.MIENF import MIENF

estimator = MIENF(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

MIENF (K=1), joint learning True 

finished: t= 0 loss= 5955.5234375 loss val= 5977.849609375 best val loss= 5977.849609375 best t= 0
finished: t= 101 loss= -7.340118408203125 loss val= -5.920987129211426 best val loss= -6.050806045532227 best t= 100
finished: t= 202 loss= -13.493276596069336 loss val= -12.519180297851562 best val loss= -12.75471305847168 best t= 196
finished: t= 303 loss= -14.800590515136719 loss val= -13.02474594116211 best val loss= -13.133474349975586 best t= 283
finished: t= 404 loss= -15.82225227355957 loss val= -12.628679275512695 best val loss= -13.133474349975586 best t= 283


true MI: 32.75224627896869
est MI: 28.301055908203125
